In [14]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate
from sklearn.model_selection import GridSearchCV

from sklearn.ensemble import HistGradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
from sklearn.model_selection import cross_val_score


In [3]:
df = pd.read_csv("data/feature_engineered_data.csv")

In [4]:
X = df.drop("loan_status", axis=1)
y = df["loan_status"]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [6]:
model = HistGradientBoostingClassifier(
    random_state=42
)

In [7]:
scores = cross_validate(
    model,
    X_train,
    y_train,
    cv=5,
    scoring=[
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc"
    ]
)

print("Accuracy :", scores["test_accuracy"].mean())
print("Precision:", scores["test_precision"].mean())
print("Recall   :", scores["test_recall"].mean())
print("F1       :", scores["test_f1"].mean())
print("ROC AUC  :", scores["test_roc_auc"].mean())

Accuracy : 0.9301549708151942
Precision: 0.8856113562102784
Recall   : 0.7875000000000001
F1       : 0.8336730507264146
ROC AUC  : 0.9767293353794868


In [8]:
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""Categorical"" and ""Enum"" are considered to be categorical features. The input must be a dataframe that is supported by narwhals (or supports it): :func:`narwhals.from_native` must work. This is the case, for instance, for pandas and polars DataFrames.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing val

In [9]:
prediction = model.predict(X_test)

probability = model.predict_proba(X_test)

In [10]:
print("Accuracy :", accuracy_score(y_test, prediction))
print("Precision:", precision_score(y_test, prediction))
print("Recall   :", recall_score(y_test, prediction))
print("F1 Score :", f1_score(y_test, prediction))
print("ROC AUC  :", roc_auc_score(y_test, probability[:,1]))

Accuracy : 0.9295477275252806
Precision: 0.8807134894091416
Recall   : 0.79
F1 Score : 0.8328940432261466
ROC AUC  : 0.9761159808544078


In [11]:
print(classification_report(y_test, prediction))

              precision    recall  f1-score   support

           0       0.94      0.97      0.96      6999
           1       0.88      0.79      0.83      2000

    accuracy                           0.93      8999
   macro avg       0.91      0.88      0.89      8999
weighted avg       0.93      0.93      0.93      8999



In [12]:
cm = confusion_matrix(y_test, prediction)

print(cm)

[[6785  214]
 [ 420 1580]]


In [15]:
learning_rates = [0.01, 0.05, 0.1, 0.2]

results = []

for lr in learning_rates:

    model = HistGradientBoostingClassifier(
        learning_rate=lr,
        random_state=42
    )

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="f1"
    )

    results.append({
        "Learning Rate": lr,
        "Mean F1": score.mean(),
        "Std": score.std()
    })

comparison = pd.DataFrame(results)

comparison.sort_values(
    by="Mean F1",
    ascending=False,
    inplace=True
)

comparison

,Learning Rate,Mean F1,Std
2,0.10,0.833673,0.006133
3,0.20,0.833004,0.009757
1,0.05,0.829630,0.004201
0,0.01,0.789614,0.004052


In [16]:
max_iters = [100, 200, 300, 500]

results = []

for it in max_iters:

    model = HistGradientBoostingClassifier(
        learning_rate=0.1,
        max_iter=it,
        random_state=42
    )

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="f1"
    )

    results.append({
        "Max Iter": it,
        "Mean F1": score.mean(),
        "Std": score.std()
    })

comparison = pd.DataFrame(results)

comparison.sort_values(
    by="Mean F1",
    ascending=False,
    inplace=True
)

comparison

,Max Iter,Mean F1,Std
3,500,0.836823,0.004660
2,300,0.836823,0.004660
1,200,0.836474,0.004699
0,100,0.833673,0.006133


In [17]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score
import pandas as pd

depths = [3, 5, 7, 10, None]

results = []

for depth in depths:

    model = HistGradientBoostingClassifier(
        learning_rate=0.1,
        max_iter=300,
        max_depth=depth,
        random_state=42
    )

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="f1"
    )

    results.append({
        "Max Depth": depth,
        "Mean F1": score.mean(),
        "Std": score.std()
    })

comparison = pd.DataFrame(results)

comparison.sort_values(
    by="Mean F1",
    ascending=False,
    inplace=True
)

comparison

,Max Depth,Mean F1,Std
3,10.0,0.838606,0.003745
2,7.0,0.837466,0.005411
4,NaN,0.836823,0.004660
1,5.0,0.836372,0.005423
0,3.0,0.830534,0.001213


In [18]:
leaf_values = [10, 20, 30, 50, 100]

results = []

for leaf in leaf_values:

    model = HistGradientBoostingClassifier(
        learning_rate=0.1,
        max_iter=300,
        max_depth=10,
        min_samples_leaf=leaf,
        random_state=42
    )

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="f1"
    )

    results.append({
        "Min Samples Leaf": leaf,
        "Mean F1": score.mean(),
        "Std": score.std()
    })

comparison = pd.DataFrame(results)

comparison.sort_values(
    by="Mean F1",
    ascending=False,
    inplace=True
)

comparison

,Min Samples Leaf,Mean F1,Std
1,20,0.838606,0.003745
4,100,0.838469,0.004635
3,50,0.836474,0.006468
0,10,0.835764,0.005938
2,30,0.835688,0.006507


In [19]:
l2_values = [0, 0.01, 0.1, 1, 5]

results = []

for l2 in l2_values:

    model = HistGradientBoostingClassifier(
        learning_rate=0.1,
        max_iter=300,
        max_depth=10,
        min_samples_leaf=20,
        l2_regularization=l2,
        random_state=42
    )

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="f1"
    )

    results.append({
        "L2 Regularization": l2,
        "Mean F1": score.mean(),
        "Std": score.std()
    })

comparison = pd.DataFrame(results)

comparison.sort_values(
    by="Mean F1",
    ascending=False,
    inplace=True
)

comparison

,L2 Regularization,Mean F1,Std
0,0.00,0.838606,0.003745
2,0.10,0.838388,0.005519
4,5.00,0.837614,0.004918
1,0.01,0.836315,0.006583
3,1.00,0.834748,0.007376


In [20]:
leaf_nodes = [15, 31, 63, 127]

results = []

for nodes in leaf_nodes:

    model = HistGradientBoostingClassifier(
        learning_rate=0.1,
        max_iter=300,
        max_depth=10,
        min_samples_leaf=20,
        max_leaf_nodes=nodes,
        random_state=42
    )

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="f1"
    )

    results.append({
        "Max Leaf Nodes": nodes,
        "Mean F1": score.mean(),
        "Std": score.std()
    })

comparison = pd.DataFrame(results)
comparison.sort_values(
    by="Mean F1",
    ascending=False,
    inplace=True
)

comparison

,Max Leaf Nodes,Mean F1,Std
1,31,0.838606,0.003745
3,127,0.835076,0.006084
2,63,0.834774,0.004917
0,15,0.833833,0.004227


In [21]:
bins = [64, 128, 255]

results = []

for b in bins:

    model = HistGradientBoostingClassifier(
        learning_rate=0.1,
        max_iter=300,
        max_depth=10,
        min_samples_leaf=20,
        max_leaf_nodes=31,
        max_bins=b,
        random_state=42
    )

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="f1"
    )

    results.append({
        "Max Bins": b,
        "Mean F1": score.mean(),
        "Std": score.std()
    })

comparison = pd.DataFrame(results)
comparison.sort_values(
    by="Mean F1",
    ascending=False,
    inplace=True
)

comparison

,Max Bins,Mean F1,Std
2,255,0.838606,0.003745
1,128,0.830431,0.006924
0,64,0.825945,0.006392


In [22]:
from sklearn.ensemble import HistGradientBoostingClassifier

final_model = HistGradientBoostingClassifier(
    learning_rate=0.1,
    max_iter=300,
    max_depth=10,
    min_samples_leaf=20,
    l2_regularization=0.0,
    max_leaf_nodes=31,
    max_bins=255,
    random_state=42
)

final_model.fit(X_train, y_train)

,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",300
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",10
,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""Categorical"" and ""Enum"" are considered to be categorical features. The input must be a dataframe that is supported by narwhals (or supports it): :func:`narwhals.from_native` must work. This is the case, for instance, for pandas and polars DataFrames.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing value

In [23]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

prediction = final_model.predict(X_test)

print(classification_report(y_test, prediction))

print("Accuracy :", accuracy_score(y_test, prediction))
print("Precision:", precision_score(y_test, prediction))
print("Recall   :", recall_score(y_test, prediction))
print("F1 Score :", f1_score(y_test, prediction))
print("ROC AUC  :", roc_auc_score(
    y_test,
    final_model.predict_proba(X_test)[:,1]
))

print(confusion_matrix(y_test,prediction))

              precision    recall  f1-score   support

           0       0.94      0.97      0.96      6999
           1       0.88      0.80      0.84      2000

    accuracy                           0.93      8999
   macro avg       0.91      0.89      0.90      8999
weighted avg       0.93      0.93      0.93      8999

Accuracy : 0.932103567063007
Precision: 0.8826446280991735
Recall   : 0.801
F1 Score : 0.8398427260812582
ROC AUC  : 0.9764559222746106
[[6786  213]
 [ 398 1602]]


In [24]:
sample = X_test.iloc[[0]].copy()

results = []

for ratio in np.arange(0.05, 0.75, 0.05):

    temp = sample.copy()

    temp["loan_percent_income"] = ratio

    probability = final_model.predict_proba(temp)[0][1]

    prediction = final_model.predict(temp)[0]

    results.append({
        "Ratio": ratio,
        "Probability": round(probability, 4),
        "Prediction": prediction
    })

pd.DataFrame(results)

,Ratio,Probability,Prediction
0,0.05,0.7114,1
1,0.10,0.6852,1
2,0.15,0.7002,1
3,0.20,0.7838,1
4,0.25,0.9318,1
5,0.30,0.9955,1
6,0.35,0.9958,1
7,0.40,0.9970,1
8,0.45,0.9957,1
9,0.50,0.9960,1
